# Q2: vitamin D and inflammation

**Question:** How is measured vitamin D associated with baseline/pre-index CRP before and after adjustment?

**Inputs:** Prepared cohort, authorized CRP measurements, and preconstructed model covariate files.

**Outputs:** Paired-sample correlations, log-log regressions, adjusted models and their related plots.

Read the CRP cleaning and pairing functions in shared/q1q2_common.py. The retained main selection uses nearest same-day/pre-index CRP; it is not the pending Q3 30-day extension. Supporting Q2 30-/90-day sensitivity is in the quality notebook. Check model status and actual included covariates when optional input files are absent.


## Configuration and paired sample


In [ ]:
from pathlib import Path
import os
import sys

if os.environ.get("CAPSTONE_PROJECT_ROOT"):
    PROJECT_ROOT = Path(os.environ["CAPSTONE_PROJECT_ROOT"]).expanduser().resolve()
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "shared" / "q1q2_common.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the example folder, or set CAPSTONE_PROJECT_ROOT.")
sys.path.insert(0, str(PROJECT_ROOT / "shared"))
from q1q2_common import *

base = load_q1_analysis_base()
baseline_crp, _ = run_q2_crp_build(base)
pair = build_q2_pair(base, baseline_crp)


## Correlation and regression definitions


In [ ]:
def run_q2_correlation(pair: pd.DataFrame) -> pd.DataFrame:
    n = len(pair)
    specs = [
        ("spearman_raw", pair["vitd_main"], pair["crp_mgL"], stats.spearmanr),
        ("spearman_log_log", pair["log_vitd"], pair["log_crp"], stats.spearmanr),
        ("pearson_raw", pair["vitd_main"], pair["crp_mgL"], stats.pearsonr),
        ("pearson_vitd_logcrp", pair["vitd_main"], pair["log_crp"], stats.pearsonr),
        ("pearson_log_log", pair["log_vitd"], pair["log_crp"], stats.pearsonr),
    ]
    corr = pd.DataFrame(
        [{"method": name, "r": func(x, y)[0], "p_value": func(x, y)[1], "n": n} for name, x, y, func in specs]
    )

    slope, intercept, _, p_value, se = stats.linregress(pair["log_vitd"], pair["log_crp"])
    result = pd.DataFrame(
        [
            {
                "n": n,
                "intercept": intercept,
                "loglog_slope": slope,
                "loglog_se": se,
                "loglog_ci_low": slope - 1.96 * se,
                "loglog_ci_high": slope + 1.96 * se,
                "loglog_p": p_value,
                "pearson_loglog": corr.loc[corr["method"] == "pearson_log_log", "r"].iloc[0],
                "spearman_raw": corr.loc[corr["method"] == "spearman_raw", "r"].iloc[0],
            }
        ]
    )
    corr.to_csv(table_path("q2_correlation_table.csv"), index=False)
    result.to_csv(table_path("q2_loglog_results.csv"), index=False)

    fig, ax = plt.subplots(figsize=(8.6, 6))
    ax.scatter(pair["vitd_main"], pair["crp_mgL"], s=4, alpha=0.10, color=WCM_RED, edgecolors="none")
    ax.set_xscale("log")
    ax.set_yscale("log")
    xs = np.linspace(pair["log_vitd"].min(), pair["log_vitd"].max(), 200)
    resid = pair["log_crp"] - (intercept + slope * pair["log_vitd"])
    s_err = np.sqrt((resid**2).sum() / (n - 2))
    xm = pair["log_vitd"].mean()
    sxx = ((pair["log_vitd"] - xm) ** 2).sum()
    ci = 1.96 * s_err * np.sqrt(1 / n + (xs - xm) ** 2 / sxx)
    ax.plot(np.exp(xs), np.exp(intercept + slope * xs), color=WCM_NAVY, lw=2.2, label="Log-log linear fit")
    ax.fill_between(
        np.exp(xs),
        np.exp(intercept + slope * xs - ci),
        np.exp(intercept + slope * xs + ci),
        color=WCM_NAVY,
        alpha=0.22,
        label="95% CI",
    )
    ax.set_xlabel("Baseline serum 25(OH)D, ng/mL (log scale)", fontsize=12)
    ax.set_ylabel("C-reactive protein, mg/L (log scale)", fontsize=12)
    ax.set_title(
        f"Log-log association between vitamin D and CRP (n = {n:,})\n"
        f"Pearson r = {result['pearson_loglog'].iloc[0]:.3f}, "
        f"Spearman rho = {result['spearman_raw'].iloc[0]:.3f}, "
        f"slope = {slope:.3f}",
        fontsize=11.5,
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, which="both")
    plt.tight_layout()
    plt.savefig(figure_path("q2_loglog_scatter.png"), dpi=170, bbox_inches="tight")
    plt.show()

    print_section("Q2 log-log correlation")
    print(corr.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print(result.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q2_correlation_table.csv, q2_loglog_results.csv, q2_loglog_scatter.png")
    return result

def run_q2_multivariable(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    d = pair.merge(base[["person_id", "index_date", "age_at_index", "female", "bmi_full"]], on="person_id", how="inner")
    d["crp_date"] = d["index_date"] - pd.to_timedelta(d["days_before"], unit="D")
    month = d["crp_date"].dt.month
    d["winter"] = ((month >= 11) | (month <= 4)).astype(int)
    d = d[(d["bmi_full"] >= 12) & (d["bmi_full"] <= 70)].copy()
    model_df = d[["log_crp", "log_vitd", "age_at_index", "female", "winter", "bmi_full"]].dropna()

    formulas = [
        ("M0 unadjusted", "log_crp ~ log_vitd"),
        ("M1 age + sex", "log_crp ~ log_vitd + age_at_index + female"),
        ("M2 age + sex + winter", "log_crp ~ log_vitd + age_at_index + female + winter"),
        ("M3 age + sex + BMI", "log_crp ~ log_vitd + age_at_index + female + bmi_full"),
        ("M4 age + sex + winter + BMI", "log_crp ~ log_vitd + age_at_index + female + winter + bmi_full"),
    ]
    rows = []
    for label, formula in formulas:
        model = smf.ols(formula, data=model_df).fit()
        ci = model.conf_int().loc["log_vitd"]
        rows.append(
            {
                "model": label,
                "n": int(model.nobs),
                "slope": model.params["log_vitd"],
                "ci_low": ci.iloc[0],
                "ci_high": ci.iloc[1],
                "p_value": model.pvalues["log_vitd"],
                "r2": model.rsquared,
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(table_path("q2_multivariable_loglog.csv"), index=False)
    print_section("Q2 multivariable log-log OLS")
    print(table.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q2_multivariable_loglog.csv")
    return table

def run_q2_quartile_plot(pair: pd.DataFrame) -> pd.DataFrame:
    df = pair.copy()
    df["vitd_quartile_num"] = pd.qcut(df["vitd_main"], 4, labels=False, duplicates="drop")
    quartiles = sorted(df["vitd_quartile_num"].dropna().astype(int).unique())

    rows = []
    data = []
    tick_labels = []
    display_labels = []
    for quartile in quartiles:
        sub = df[df["vitd_quartile_num"] == quartile]
        crp_values = sub["crp_mgL"].dropna().values
        vitd_values = sub["vitd_main"].dropna()
        if len(crp_values) == 0 or vitd_values.empty:
            continue
        label = f"Q{quartile + 1}"
        if quartile == min(quartiles):
            display_label = "Q1 lowest"
        elif quartile == max(quartiles):
            display_label = f"Q{quartile + 1} highest"
        else:
            display_label = label
        data.append(crp_values)
        display_labels.append(display_label)
        tick_labels.append(f"{label}\n{vitd_values.min():.0f}-{vitd_values.max():.0f} ng/mL\n(n={len(sub):,})")
        rows.append(
            {
                "vitd_quartile": display_label,
                "n": len(sub),
                "vitd_min": vitd_values.min(),
                "vitd_max": vitd_values.max(),
                "crp_median": np.median(crp_values),
                "crp_q1": np.percentile(crp_values, 25),
                "crp_q3": np.percentile(crp_values, 75),
            }
        )
    if len(data) < 2:
        raise ValueError("Need at least two vitamin D quartiles with CRP values for Q2 quartile plot")
    kw = stats.kruskal(*data)
    summary = pd.DataFrame(rows)
    summary["kruskal_p"] = kw.pvalue
    summary.to_csv(table_path("q2_crp_by_vitd_quartile_summary.csv"), index=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    try:
        bp = ax.boxplot(data, tick_labels=tick_labels, showfliers=False, patch_artist=True, widths=0.55)
    except TypeError:
        bp = ax.boxplot(data, labels=tick_labels, showfliers=False, patch_artist=True, widths=0.55)
    for box in bp["boxes"]:
        box.set_facecolor("#f3d6d6")
        box.set_edgecolor(WCM_NAVY)
        box.set_linewidth(1.2)
    for median in bp["medians"]:
        median.set_color(WCM_RED)
        median.set_linewidth(2.4)
    for element in bp["whiskers"] + bp["caps"]:
        element.set_color(WCM_NAVY)
    ax.set_yscale("log")
    ax.set_xlabel("Baseline serum 25(OH)D quartile", fontsize=12)
    ax.set_ylabel("C-reactive protein, mg/L (log scale)", fontsize=12)
    ax.set_title(f"CRP by vitamin D quartile (n = {len(pair):,})\nKruskal-Wallis p = {kw.pvalue:.1e}", fontsize=11.5)
    for idx, values in enumerate(data, 1):
        ax.text(idx, np.median(values) * 1.28, f"{np.median(values):.2f}", ha="center", fontsize=11, color=WCM_RED, fontweight="bold")
    ax.grid(True, alpha=0.3, axis="y", which="both")
    plt.tight_layout()
    plt.savefig(figure_path("q2_crp_by_vitd_quartile.png"), dpi=170, bbox_inches="tight")
    plt.savefig(figure_path("q2_crp_by_vitd_quartile_labeled.png"), dpi=170, bbox_inches="tight")
    plt.show()

    print_section("Q2 CRP by vitamin D quartile")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q2_crp_by_vitd_quartile_summary.csv, q2_crp_by_vitd_quartile.png, and q2_crp_by_vitd_quartile_labeled.png")
    return summary


## Adjusted model definitions


In [ ]:
def find_q2_covariate_file(filename: str) -> Path | None:
    for path in [Path(filename), table_path(filename), OUTPUT_DIR / filename]:
        if path.exists():
            return path
    return None


def read_q2_model_covariate_table(filename: str, columns: list[str]) -> pd.DataFrame:
    path = find_q2_covariate_file(filename)
    if path is None:
        raise FileNotFoundError(f"Required Q2 covariate table not found: {filename}")
    table = pd.read_csv(path)
    if "person_id" not in table.columns:
        raise ValueError(f"{filename} is missing person_id")
    table = as_person_id(table)
    keep = [col for col in columns if col in table.columns]
    missing = [col for col in columns if col not in table.columns]
    if missing:
        raise ValueError(f"{filename} missing required columns: {missing}")
    return table[keep].drop_duplicates("person_id")


def prepare_q2_extended_model_df(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    base_cols = [
        "person_id",
        "index_date",
        "age_at_index",
        "female",
        "gender",
        "bmi_full",
        "race_eth_simplified",
        "race_black",
        "race_hispanic",
        "race_asian",
        "race_other",
        "baseline_diabetes",
    ]
    d = pair.merge(base[[col for col in base_cols if col in base.columns]], on="person_id", how="inner")
    d = normalize_index_date(d)
    d["crp_date"] = d["index_date"] - pd.to_timedelta(d["days_before"], unit="D")
    month = d["crp_date"].dt.month
    d["winter"] = ((month >= 11) | (month <= 4)).astype(int)
    d["bmi_full"] = pd.to_numeric(d["bmi_full"], errors="coerce")
    d = d[(d["bmi_full"] >= 12) & (d["bmi_full"] <= 70)].copy()

    if "female" not in d.columns and "gender" in d.columns:
        d["female"] = (d["gender"] == "Female").astype(int)
    if "race_black" not in d.columns and "race_eth_simplified" in d.columns:
        race = d["race_eth_simplified"]
        d["race_black"] = (race == "Black or African American").astype(int)
        d["race_hispanic"] = (race == "Hispanic or Latino").astype(int)
        d["race_asian"] = (race == "Asian").astype(int)
        d["race_other"] = race.isin(["Other/Multiracial", "Unknown/Other"]).astype(int)

    merge_specs = [
        ("q2_income_band.csv", ["person_id", "income_band"]),
        ("q2_education.csv", ["person_id", "education_level"]),
        ("q2_insurance.csv", ["person_id", "insurance_status"]),
        (
            "q2_medication_use.csv",
            [
                "person_id",
                "vitd_supplement_use",
                "vitd_supplement_120d",
                "vitd_supplement_active_on_target",
                "vitd_supplement_history_pre_target",
                "active_vitd_analog_use_120d",
            ],
        ),
    ]
    for filename, columns in merge_specs:
        try:
            covariates = read_q2_model_covariate_table(filename, columns)
        except FileNotFoundError as exc:
            print(f"Extended model covariate skipped: {exc}")
            continue
        d = d.merge(covariates, on="person_id", how="left")

    if "vitd_supplement_use" in d.columns:
        d["vitd_supplement_use"] = pd.to_numeric(d["vitd_supplement_use"], errors="coerce").fillna(0).astype(int)

    d.to_csv(table_path("q2_extended_model_dataset.csv"), index=False)
    return d


def classify_q2_model_term(term: str) -> str:
    if term == "Intercept":
        return "intercept"
    if term == "log_vitd":
        return "exposure"
    if term in {"age_at_index", "female", "winter", "bmi_full"}:
        return "core_covariate"
    if term.startswith("race_") or term == "baseline_diabetes":
        return "extended_covariate"
    if term.startswith("C(income_band)") or term.startswith("C(education_level)") or term.startswith("C(insurance_status)"):
        return "SES_covariate"
    if term == "vitd_supplement_use":
        return "supplement_covariate"
    return "other_covariate"


def fit_q2_extended_ols_rows(model_df: pd.DataFrame, model_specs: list[dict[str, object]]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows: list[dict[str, object]] = []
    coefficient_rows: list[dict[str, object]] = []
    for spec in model_specs:
        label = str(spec["model"])
        formula = str(spec["formula"])
        fields = list(spec["fields"])
        missing_fields = [field for field in fields if field not in model_df.columns]
        if missing_fields:
            rows.append(
                {
                    "model": label,
                    "status": "missing_fields",
                    "n": 0,
                    "formula": formula,
                    "complete_case_fields": ", ".join(fields),
                    "slope": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "p_value": np.nan,
                    "r2": np.nan,
                    "error": f"Missing fields: {missing_fields}",
                }
            )
            continue
        d = model_df[fields].dropna().copy()
        if d.empty:
            rows.append(
                {
                    "model": label,
                    "status": "empty_complete_case",
                    "n": 0,
                    "formula": formula,
                    "complete_case_fields": ", ".join(fields),
                    "slope": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "p_value": np.nan,
                    "r2": np.nan,
                    "error": "No complete-case rows",
                }
            )
            continue
        try:
            model = smf.ols(formula, data=d).fit()
            coefficient_ci = model.conf_int()
            for term in model.params.index:
                coefficient_rows.append(
                    {
                        "model": label,
                        "status": "fit",
                        "n": int(model.nobs),
                        "formula": formula,
                        "term": term,
                        "term_role": classify_q2_model_term(term),
                        "estimate": model.params[term],
                        "std_error": model.bse[term],
                        "ci_low": coefficient_ci.loc[term].iloc[0],
                        "ci_high": coefficient_ci.loc[term].iloc[1],
                        "p_value": model.pvalues[term],
                        "r2": model.rsquared,
                    }
                )
            ci = model.conf_int().loc["log_vitd"]
            rows.append(
                {
                    "model": label,
                    "status": "fit",
                    "n": int(model.nobs),
                    "formula": formula,
                    "complete_case_fields": ", ".join(fields),
                    "slope": model.params["log_vitd"],
                    "ci_low": ci.iloc[0],
                    "ci_high": ci.iloc[1],
                    "p_value": model.pvalues["log_vitd"],
                    "r2": model.rsquared,
                    "error": "",
                }
            )
        except Exception as exc:
            rows.append(
                {
                    "model": label,
                    "status": "failed",
                    "n": len(d),
                    "formula": formula,
                    "complete_case_fields": ", ".join(fields),
                    "slope": np.nan,
                    "ci_low": np.nan,
                    "ci_high": np.nan,
                    "p_value": np.nan,
                    "r2": np.nan,
                    "error": str(exc),
                }
            )
    return pd.DataFrame(rows), pd.DataFrame(coefficient_rows)


def plot_q2_extended_logvitd_forest(table: pd.DataFrame) -> None:
    plot_data = table[(table["status"] == "fit") & table["slope"].notna()].copy()
    if plot_data.empty:
        print("Skipped q2_extended_model_logvitd_forest.png because no fitted extended models were available.")
        return

    plot_data["label"] = plot_data["model"].astype(str) + " (n=" + plot_data["n"].astype(int).map("{:,}".format) + ")"
    y = np.arange(len(plot_data))[::-1]
    x = plot_data["slope"].to_numpy(dtype=float)
    lower = plot_data["ci_low"].to_numpy(dtype=float)
    upper = plot_data["ci_high"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(8.5, max(3.2, 0.62 * len(plot_data) + 1.5)))
    ax.errorbar(
        x,
        y,
        xerr=[x - lower, upper - x],
        fmt="o",
        color=WCM_RED,
        ecolor=WCM_NAVY,
        elinewidth=2,
        capsize=4,
        markersize=6,
    )
    ax.axvline(0, color="0.35", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(plot_data["label"])
    ax.set_xlabel("Coefficient for log 25(OH)D in log-CRP model")
    ax.set_title("Q2 extended models: log vitamin D association with log CRP")
    ax.grid(axis="x", alpha=0.25)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    plt.savefig(figure_path("q2_extended_model_logvitd_forest.png"), dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def q2_reference_value_for_adjusted_plot(d: pd.DataFrame, field: str) -> object:
    if field in {"age_at_index", "bmi_full"}:
        return pd.to_numeric(d[field], errors="coerce").median()
    if field in {"race_black", "race_hispanic", "race_asian", "race_other", "baseline_diabetes", "vitd_supplement_use"}:
        return 0
    if field == "winter":
        return 0 if (d[field] == 0).any() else d[field].dropna().mode().iloc[0]
    if field == "income_band":
        return "Income 35-100k" if (d[field] == "Income 35-100k").any() else d[field].dropna().mode().iloc[0]
    if field == "education_level":
        return "College or higher" if (d[field] == "College or higher").any() else d[field].dropna().mode().iloc[0]
    if field == "insurance_status":
        return "Insured" if (d[field] == "Insured").any() else d[field].dropna().mode().iloc[0]
    mode = d[field].dropna().mode()
    return mode.iloc[0] if not mode.empty else np.nan


def q2_adjusted_prediction_grid(d: pd.DataFrame, fields: list[str], x_grid: np.ndarray) -> pd.DataFrame:
    grid = pd.DataFrame({"log_vitd": x_grid})
    for field in fields:
        if field in {"log_crp", "log_vitd"}:
            continue
        if field in d.columns:
            grid[field] = q2_reference_value_for_adjusted_plot(d, field)
    return grid


def plot_q2_extended_adjusted_fit_lines(model_df: pd.DataFrame, model_specs: list[dict[str, object]]) -> None:
    scatter = model_df[["vitd_main", "crp_mgL", "log_vitd"]].dropna().copy()
    scatter = scatter[(scatter["vitd_main"] > 0) & (scatter["crp_mgL"] > 0)]
    if scatter.empty:
        print("Skipped q2_extended_model_adjusted_fit_lines.png because no positive vitamin D/CRP values were available.")
        return

    x_low, x_high = scatter["log_vitd"].quantile([0.01, 0.99])
    if not np.isfinite(x_low) or not np.isfinite(x_high) or x_low >= x_high:
        x_low, x_high = scatter["log_vitd"].min(), scatter["log_vitd"].max()
    x_grid = np.linspace(x_low, x_high, 120)

    fitted_panels: list[dict[str, object]] = []
    for spec in model_specs:
        label = str(spec["model"])
        formula = str(spec["formula"])
        fields = list(spec["fields"])
        if any(field not in model_df.columns for field in fields):
            continue
        d = model_df[fields].dropna().copy()
        if d.empty:
            continue
        try:
            model = smf.ols(formula, data=d).fit()
            grid = q2_adjusted_prediction_grid(d, fields, x_grid)
            pred = model.get_prediction(grid).summary_frame(alpha=0.05)
            fitted_panels.append(
                {
                    "model": label,
                    "n": int(model.nobs),
                    "x": np.exp(x_grid),
                    "mean": np.exp(pred["mean"].to_numpy(dtype=float)),
                    "low": np.exp(pred["mean_ci_lower"].to_numpy(dtype=float)),
                    "high": np.exp(pred["mean_ci_upper"].to_numpy(dtype=float)),
                }
            )
        except Exception as exc:
            print(f"Skipped adjusted fitted line for {label}: {exc}")

    if not fitted_panels:
        print("Skipped q2_extended_model_adjusted_fit_lines.png because no adjusted fitted lines could be generated.")
        return

    panel_count = len(fitted_panels)
    fig, axes = plt.subplots(1, panel_count, figsize=(5.1 * panel_count, 4.2), sharex=True, sharey=True)
    if panel_count == 1:
        axes = [axes]

    rng = np.random.default_rng(714)
    max_points = min(len(scatter), 6000)
    scatter_plot = scatter.iloc[rng.choice(len(scatter), size=max_points, replace=False)] if len(scatter) > max_points else scatter

    for ax, panel in zip(axes, fitted_panels):
        ax.scatter(scatter_plot["vitd_main"], scatter_plot["crp_mgL"], s=5, alpha=0.08, color="0.25", edgecolors="none")
        ax.plot(panel["x"], panel["mean"], color=WCM_RED, lw=2.3)
        ax.fill_between(panel["x"], panel["low"], panel["high"], color=WCM_NAVY, alpha=0.14, linewidth=0)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_title(f'{panel["model"]}\n(n={panel["n"]:,})', fontsize=10)
        ax.grid(alpha=0.25, which="both")
        for spine in ["top", "right"]:
            ax.spines[spine].set_visible(False)

    axes[0].set_ylabel("CRP, mg/L (log scale)")
    for ax in axes:
        ax.set_xlabel("Serum/plasma 25(OH)D, ng/mL (log scale)")
    fig.suptitle("Q2 adjusted log-log fitted lines at reference covariate profile", y=1.04, fontsize=14)
    fig.tight_layout()
    plt.savefig(figure_path("q2_extended_model_adjusted_fit_lines.png"), dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def run_q2_extended_models(base: pd.DataFrame, pair: pd.DataFrame) -> pd.DataFrame:
    d = prepare_q2_extended_model_df(base, pair)

    core_fields = ["log_crp", "log_vitd", "age_at_index", "female", "winter", "bmi_full"]
    extended_fields = core_fields + [
        "race_black",
        "race_hispanic",
        "race_asian",
        "race_other",
        "baseline_diabetes",
        "income_band",
        "education_level",
        "insurance_status",
    ]
    supplement_fields = extended_fields + ["vitd_supplement_use"]

    model_specs = [
        {
            "model": "A_core_current",
            "formula": "log_crp ~ log_vitd + age_at_index + female + winter + bmi_full",
            "fields": core_fields,
        },
        {
            "model": "B_extended_race_diabetes_SES",
            "formula": (
                "log_crp ~ log_vitd + age_at_index + female + winter + bmi_full "
                "+ race_black + race_hispanic + race_asian + race_other "
                "+ baseline_diabetes + C(income_band) + C(education_level) + C(insurance_status)"
            ),
            "fields": extended_fields,
        },
        {
            "model": "C_extended_plus_vitd_supplement",
            "formula": (
                "log_crp ~ log_vitd + age_at_index + female + winter + bmi_full "
                "+ race_black + race_hispanic + race_asian + race_other "
                "+ baseline_diabetes + C(income_band) + C(education_level) + C(insurance_status) "
                "+ vitd_supplement_use"
            ),
            "fields": supplement_fields,
        },
    ]

    table, coefficient_table = fit_q2_extended_ols_rows(d, model_specs)
    table.to_csv(table_path("q2_extended_model_loglog.csv"), index=False)
    coefficient_table.to_csv(table_path("q2_extended_model_full_coefficients.csv"), index=False)
    plot_q2_extended_logvitd_forest(table)
    plot_q2_extended_adjusted_fit_lines(d, model_specs)

    print_section("Q2 extended log-log OLS models")
    print(table[["model", "status", "n", "slope", "ci_low", "ci_high", "p_value", "r2"]].to_string(index=False, float_format=lambda x: f"{x:.4g}"))
    print("Saved q2_extended_model_dataset.csv, q2_extended_model_loglog.csv, q2_extended_model_full_coefficients.csv, q2_extended_model_logvitd_forest.png, and q2_extended_model_adjusted_fit_lines.png")
    return table


## Fit and summarize


In [ ]:
run_q2_correlation(pair)
run_q2_multivariable(base, pair)
run_q2_extended_models(base, pair)
run_q2_quartile_plot(pair)
